In [1]:
def pretty_print(docs):
    print(f"\n{'-' * 100}\n".join(
        [
            f"Document {i+1}:\n\n{d.page_content}\nMetadata: {d.metadata}"
            for i, d in enumerate(docs)
        ]
    )
)

In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.embeddings import 
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import os

In [7]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2488.68it/s]


In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [12]:
path = "G:\\langc-in-production\\all_document\\state_of_the_union.txt"

In [14]:
document = TextLoader(path, encoding="utf-8").load()

In [18]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [19]:
texts = text_splitter.split_documents(documents=document)

In [20]:
print(len(texts))

96


In [21]:
for id, text in enumerate(texts):
    text.metadata['id'] = id

In [32]:
retriever = FAISS.from_documents(texts, embedding_model).as_retriever(search_kwargs={'k': 10})

In [33]:
query = ("what did the president say about ketanji brown jackson")

In [34]:
docs = retriever.invoke(query)

In [35]:
print(docs)

[Document(id='1f861b2e-eaf8-4eba-aa15-4b7b7185e787', metadata={'source': 'G:\\langc-in-production\\all_document\\state_of_the_union.txt', 'id': 73}, page_content='One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'), Document(id='15141b63-1ccc-4af0-88d2-91360900586c', metadata={'source': 'G:\\langc-in-production\\all_document\\state_of_the_union.txt', 'id': 72}, page_content='We cannot let this happen. \n\nTonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen 

In [36]:
print(docs)

[Document(id='1f861b2e-eaf8-4eba-aa15-4b7b7185e787', metadata={'source': 'G:\\langc-in-production\\all_document\\state_of_the_union.txt', 'id': 73}, page_content='One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'), Document(id='15141b63-1ccc-4af0-88d2-91360900586c', metadata={'source': 'G:\\langc-in-production\\all_document\\state_of_the_union.txt', 'id': 72}, page_content='We cannot let this happen. \n\nTonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen 

In [37]:
pretty_print(docs)

Document 1:

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.
Metadata: {'source': 'G:\\langc-in-production\\all_document\\state_of_the_union.txt', 'id': 73}
----------------------------------------------------------------------------------------------------
Document 2:

We cannot let this happen. 

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justic

In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import FlashrankRerank
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, LLMChainFilter

In [40]:
compressor = FlashrankRerank()

INFO:flashrank.Ranker:Downloading ms-marco-MultiBERT-L-12...
ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:30<00:00, 3.36MiB/s]  


In [49]:
compressor_2 = LLMChainExtractor.from_llm(llm)

In [ ]:
compressor_3 = LLMChainFilter.from_llm(llm)

In [41]:
compress_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)

In [50]:
compressor_retriever_2 = ContextualCompressionRetriever(base_compressor=compressor_2, base_retriever=retriever)

In [43]:
query

'what did the president say about ketanji brown jackson'

In [42]:
compress_retriever.invoke(query)

[Document(metadata={'id': 73, 'relevance_score': np.float32(0.999181), 'source': 'G:\\langc-in-production\\all_document\\state_of_the_union.txt'}, page_content='One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'),
 Document(metadata={'id': 2, 'relevance_score': np.float32(0.99533266), 'source': 'G:\\langc-in-production\\all_document\\state_of_the_union.txt'}, page_content='He met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world. \n\nGroups of citizens blocking tanks with their bodies. Everyone from students to retirees teachers turned soldiers defending their homeland. \n\nIn this struggle as President Zelensky

In [51]:
compressor_retriever_2.invoke(query)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[Document(metadata={'source': 'G:\\langc-in-production\\all_document\\state_of_the_union.txt', 'id': 73}, page_content='And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'),
 Document(metadata={'source': 'G:\\langc-in-production\\all_document\\state_of_the_union.txt', 'id': 74}, page_content='A former top litigator in private practice. A former federal public defender. And from a family of public school educators and police officers. A consensus builder. Since she’s been nominated, she’s received a broad range of support—from the Fraternal Order of Police to former judges appointed by Democrats and Republicans.')]

In [45]:
from langchain_classic.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(llm=llm, retriever=compress_retriever)

In [46]:
chain.invoke(query)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


{'query': 'what did the president say about ketanji brown jackson',
 'result': 'The President said that Ketanji Brown Jackson is "one of our nation\'s top legal minds" and that she will "continue Justice Breyer\'s legacy of excellence" on the United States Supreme Court, after he nominated her 4 days prior to the speech.'}

In [52]:
chain2 = RetrievalQA.from_chain_type(llm, retriever=compressor_retriever_2)

In [53]:
chain2.invoke(query)

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://ap

{'query': 'what did the president say about ketanji brown jackson',
 'result': 'The president said that Ketanji Brown Jackson is "one of our nation\'s top legal minds" and will continue Justice Breyer\'s legacy of excellence. He also mentioned that she is a former top litigator in private practice, a former federal public defender, and comes from a family of public school educators and police officers. Additionally, he noted that she is a consensus builder who has received a broad range of support from various groups, including the Fraternal Order of Police and former judges appointed by both Democrats and Republicans.'}